# Integrate Glean with Astra DB

> Demo showing how to index [Astra DB](https://docs.datastax.com/en/astra-db-serverless/index.html) data into Glean.

This notebook is a walkthrough explaining how to use an Astra DB collection as a data source for Glean.

Using the Python Data API client, we'll read from a collection and use the glean `indexingAPI` through a `Datasource` to index the collection contents in Glean.

## 1. Set up Astra DB


ℹ️ See the [Astra Reference documentation](https://docs.datastax.com/en/astra-db-serverless/databases/create-database.html).

### 1.1: Create an Astra account

Access [https://astra.datastax.com](https://astra.datastax.com) and register with `Google` or `Github` account.

![](https://github.com/datastaxdevs/mini-demo-astradb-glean/blob/main/images/01-login.png?raw=true)

### 1.2: Create a Database in Astra DB

Get to the databases dashboard (by clicking on Databases in the left-hand navigation bar, expanding it if necessary), and click the `[Create Database]` button on the right.

![](https://github.com/datastaxdevs/mini-demo-astradb-glean/blob/main/images/02-create-db.png?raw=true)


**ℹ️ Field Description**

| Field                                      | Description                                                                                                                                                                                                                                   |
|--------------------------------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **Vector Database vs Serverless Database** | Choose `Vector Database`. In june 2023, Cassandra introduced the support of vector search to enable Generative AI use cases.                                                                                                                   |
| **Database name**                          | Database names are permanent. They must start and end with a letter or number, and they can contain no more than 50 characters, including letters, numbers, and the special characters `& + - _ ( ) < > . , @`. It is recommended to have a database for each of your applications. The free tier is limited to 5 databases. |
| **Cloud Provider**                         | Choose whatever you like. Click a cloud provider logo, pick an Area in the list and finally pick a region. We recommend choosing a region that is closest to you to reduce latency. In the free tier, there is very little difference.            |
| **Cloud Region**                           | Pick a region close to you, among those available for the selected cloud provider and your plan.      

If all fields are filled properly, clicking the "Create Database" button will start the process.

![](https://github.com/datastaxdevs/mini-demo-astradb-glean/blob/main/images/03-pending-db.png?raw=true)

It should take a couple of minutes for your database to become `Active`.

![](https://github.com/datastaxdevs/mini-demo-astradb-glean/blob/main/images/04-active-db.png?raw=true)

### 1.3: Create an Astra database token

To [connect to your database](https://docs.datastax.com/en/astra-db-serverless/get-started/quickstart.html#create-a-database-and-store-your-credentials), you need the **API endpoint** and a **Database token**.

The API endpoint is available on the database screen, there is a little icon to copy the URL in your clipboard. (it should look like `https://<db-id>-<db-region>.apps.astra.datastax.com`).

![](https://github.com/datastaxdevs/mini-demo-astradb-glean/blob/main/images/05-create-token-db.png?raw=true)

To get a token click the `[Generate Token]` button on the right. It will generate a token that you can copy to your clipboard.

## 2. Obtain a Glean token

> [Glean Documentation](https://developers.glean.com/docs/indexing_api/indexing_api_tokens/)

Admins can manage Glean API tokens via the API tokens page within Workspace Settings:

```
Workspace > Setup > API tokens > Indexing tokens tab
```

As a Glean admin, create a token and assign permissions (or have an admin do it for you).


## 3. Installation

Install the required dependencies:

In [ ]:
!pip install --quiet \
    "astrapy>=2.0,<3.0" \
    "datasets>=3.5,<4.0" \
    "https://app.glean.com/meta/indexing_api_client.zip"  # Glean does not distribute on PyPI

Import the required packages:

In [ ]:
import os

from getpass import getpass

from astrapy import DataAPIClient
from datasets import load_dataset

import glean_indexing_api_client as indexing_api
from glean_indexing_api_client.api import datasources_api, documents_api
from glean_indexing_api_client.model.custom_datasource_config import (
    CustomDatasourceConfig,
)
from glean_indexing_api_client.model.object_definition import ObjectDefinition
from glean_indexing_api_client.model.index_document_request import IndexDocumentRequest
from glean_indexing_api_client.model.document_definition import DocumentDefinition
from glean_indexing_api_client.model.content_definition import ContentDefinition
from glean_indexing_api_client.model.document_permissions_definition import (
    DocumentPermissionsDefinition,
)

## 4. Set up variables

In [ ]:
os.environ["ASTRA_DB_APPLICATION_TOKEN"] = getpass("ASTRA_DB_APPLICATION_TOKEN = ").strip()
os.environ["ASTRA_DB_API_ENDPOINT"] = input("ASTRA_DB_API_ENDPOINT = ").strip()
os.environ["ASTRA_DB_COLLECTION_NAME"] = input("ASTRA_DB_COLLECTION_NAME = ").strip() or "glean_source_collection"
os.environ["ASTRA_DB_KEYSPACE"] = input("(optional) ASTRA_DB_KEYSPACE = ").strip()

if os.environ["ASTRA_DB_KEYSPACE"] == "":
    del os.environ["ASTRA_DB_KEYSPACE"]

In [ ]:
os.environ["GLEAN_API_TOKEN"] = getpass("GLEAN_API_TOKEN = ").strip()
os.environ["GLEAN_CUSTOMER"] = input("GLEAN_CUSTOMER = ").strip()
os.environ["GLEAN_DATASOURCE_NAME"] = input("GLEAN_DATASOURCE_NAME = ").strip()

In [ ]:
ASTRA_DB_APPLICATION_TOKEN = os.environ["ASTRA_DB_APPLICATION_TOKEN"]
ASTRA_DB_API_ENDPOINT = os.environ["ASTRA_DB_API_ENDPOINT"]
ASTRA_DB_COLLECTION_NAME = os.environ["ASTRA_DB_COLLECTION_NAME"]
ASTRA_DB_KEYSPACE = os.getenv("ASTRA_DB_KEYSPACE")

GLEAN_API_TOKEN = os.environ["GLEAN_API_TOKEN"]
GLEAN_CUSTOMER = os.environ["GLEAN_CUSTOMER"]
GLEAN_DATASOURCE_NAME = os.environ["GLEAN_DATASOURCE_NAME"]

## 5. Populate Astra DB

Create an empty collection and fill it with sample data.

### 5.1: Connect to Astra DB

In [ ]:
# Initialize Astra DB client
client = DataAPIClient(callers=[("glean", "1.0")])
database = client.get_database(
    ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
    keyspace=ASTRA_DB_KEYSPACE,
)
print(f"[ OK ] - Credentials are OK, your database name is {database.name()}.")

### 5.2: Create a collection

In [ ]:
# Create collection
source_collection = database.create_collection(ASTRA_DB_COLLECTION_NAME)
print(f"[ OK ] - Collection {source_collection.name} is ready.")

### 5.3: Load dataset

In [ ]:
print(f"[INFO] - Downloading data from Hugging Face 🤗.")
philo_dataset = load_dataset("datastax/philosopher-quotes")["train"]
print(f"[ OK ] - Dataset loaded in memory.")
print(f"[INFO] - Sample record: {philo_dataset[16]}")

### 5.4 Load data into the collection

In [ ]:
def load_to_astra_db(data_to_insert, collection):
    """Load all of the provided data into a collection."""
    def split_tags(t):
        return [tag for tag in (t or "").split(";") if tag]

    documents_to_insert = [
        {
            **item,
            **{"_id": index, "tags": split_tags(item["tags"])},
        }
        for index, item in enumerate(data_to_insert)
    ]
    collection.insert_many(documents_to_insert)


# Insert documents into Astra DB
philo_count = len(philo_dataset)
print(f"[INFO] - Inserting {philo_count} documents into Astra DB...")
load_to_astra_db(philo_dataset, source_collection)
print(f"[ OK ] - Insertion finished.")

## 6. Index data in Glean

### 6.1 Initialize Glean client

In [ ]:
# Setup Glean API
GLEAN_API_ENDPOINT = f"https://{GLEAN_CUSTOMER}-be.glean.com/api/index/v1"
print(f"[INFO] - Glean API setup, endpoint is: {GLEAN_API_ENDPOINT}")

# Initialize Glean client
configuration = indexing_api.Configuration(
    host=GLEAN_API_ENDPOINT, access_token=GLEAN_API_TOKEN
)
api_client = indexing_api.ApiClient(configuration)
datasource_api = datasources_api.DatasourcesApi(api_client)
print(f"[ OK ] - Glean client initialized")

# Create and register datasource in Glean
datasource_config = CustomDatasourceConfig(
    name=GLEAN_DATASOURCE_NAME,
    display_name="AstraDB Collection DataSource",
    datasource_category="PUBLISHED_CONTENT",
    url_regex=f"^{ASTRA_DB_API_ENDPOINT}",
    object_definitions=[
        ObjectDefinition(doc_category="PUBLISHED_CONTENT", name="AstraVectorEntry")
    ],
)

try:
    datasource_api.adddatasource_post(datasource_config)
    print(f"[ OK ] - DataSource has been created!.")
except indexing_api.ApiException as e:
    print(f"[ ERROR ] - Error creating datasource: {e}.")

### 6.2 Create functions to index documents

In [ ]:
def index_astra_db_document_into_glean(astra_document):
    """Index one Astra DB document into Glean."""
    document_id = str(astra_document["_id"])
    title = f"{astra_document['author']} quote_{astra_document['_id']}"
    body_text = astra_document["quote"]
    datasource_name = GLEAN_DATASOURCE_NAME
    request = IndexDocumentRequest(
        document=DocumentDefinition(
            datasource=datasource_name,
            title=title,
            id=document_id,
            view_url=ASTRA_DB_API_ENDPOINT,
            body=ContentDefinition(mime_type="text/plain", text_content=body_text),
            permissions=DocumentPermissionsDefinition(allow_anonymous_access=True),
        )
    )
    documents_api_client = documents_api.DocumentsApi(api_client)
    try:
        documents_api_client.indexdocument_post(request)
    except indexing_api.ApiException as e:
        print(f"Error indexing document {document_id}: {e}")


def index_documents_to_glean(collection):
    """Index all documents from an Astra DB collection to Glean."""
    total_docs = collection.count_documents({}, upper_bound=1000)
    print(f"[INFO] - Indexing {total_docs} documents into Glean...")
    for doc in collection.find():
        try:
            index_astra_db_document_into_glean(doc)
        except Exception as error:
            print(f"Error indexing document {doc['_id']}: {error}")
    print(f"[ OK ] - Indexing finished.")


### 6.3 Index documents


In [ ]:
index_documents_to_glean(source_collection)

print(f"Import job completed successfully!")

## Wrap up and more information

Congratulations: you have indexed data from an Astra DB collection into Glean!

You can inspect the Astra DB collection in your Astra dashboard: navigate to the database and find the "Data explorer" tab to locate your collection.

You can perform a test with Glean: search for the content you just indexed and verify the response contains information coming from the inserted dataset.

ℹ️ [Glean integration page](https://docs.datastax.com/en/astra-db-serverless/integrations/glean.html) on Astra DB documentation.